In [0]:
dbutils.widgets.text("audit_table",        "banking_demo.bronze.transactions_data_audit")
dbutils.widgets.text("max_quarantine_pct", "5.0")

audit_table = dbutils.widgets.get("audit_table")
max_q_pct   = float(dbutils.widgets.get("max_quarantine_pct"))

latest = (
    spark.table(audit_table)
         .orderBy("run_ts", ascending=False)
         .limit(1)
         .collect()[0]
)

print("=" * 55)
print("  Audit check report")
print("=" * 55)
print(f"  Audit table:       {audit_table}")
print(f"  Run ts:            {latest['run_ts']}")
print(f"  Total rows:        {latest['total_rows']:,}")
print(f"  Good rows:         {latest['good_rows']:,}")
print(f"  Quarantine rows:   {latest['quarantine_rows']:,}")
print(f"  Quarantine pct:    {latest['quarantine_pct']:.2f}%")
print(f"  Max rate age mins: {latest['max_rate_age_mins']:.1f}")
print(f"  Stale rate count:  {latest['stale_rate_count']:,}")
print(f"  Audit status:      {latest['audit_status']}")
print("=" * 55)

assert latest["total_rows"] > 0, \
    "FATAL: audit table shows zero rows — pipeline produced no data"

assert latest["quarantine_pct"] <= max_q_pct, \
    (f"FATAL: quarantine rate {latest['quarantine_pct']:.2f}% "
     f"exceeds threshold of {max_q_pct:.2f}%")

print("\nAll checks passed. Pipeline is healthy.")